# EZ-VC RunPod Gradio UI

This notebook clones EZ-VC on RunPod, installs the runtime into `.venv`, and starts the inference or training Gradio UI with public sharing enabled.

Set `HF_TOKEN` in the RunPod environment before starting Jupyter, or paste the token into the UI field after launch.

In [ ]:
from pathlib import Path
import os

WORKDIR = Path('/workspace') if Path('/workspace').exists() else Path.cwd()
REPO_DIR = WORKDIR / 'EZ-VC'
os.environ['EZVC_REPO_URL'] = 'https://github.com/RahulBhalley/EZ-VC.git'
os.environ['EZVC_REPO_DIR'] = str(REPO_DIR)

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token loaded from environment')
else:
    print('No HF_TOKEN found in environment. You can paste it into the Gradio HF Token field after launch.')

print('Repo directory:', REPO_DIR)

## Clone and install

This cell is idempotent. Re-run it after recreating a pod or changing the branch.

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "${EZVC_REPO_DIR}")"
if [[ ! -d "${EZVC_REPO_DIR}/.git" ]]; then
  git clone --recursive "${EZVC_REPO_URL}" "${EZVC_REPO_DIR}"
else
  git -C "${EZVC_REPO_DIR}" pull --ff-only
  git -C "${EZVC_REPO_DIR}" submodule update --init --recursive
fi

cd "${EZVC_REPO_DIR}"
python3 -m venv .venv
.venv/bin/python -m pip install -U pip setuptools wheel
.venv/bin/python -m pip install -e .
.venv/bin/python -m pip install torchcodec
.venv/bin/python -m pip install --no-deps 'espnet @ git+https://github.com/wanchichen/espnet.git@ssl'
.venv/bin/python -m pip install \
  configargparse typeguard humanfriendly librosa==0.9.2 jamo h5py kaldiio \
  torch_complex nltk g2p_en espnet_tts_frontend opt-einsum editdistance \
  sentencepiece resampy inflect distance more_itertools jaconv


## Start inference UI

Run this cell to open the EZ-VC inference UI. Gradio will print a public `gradio.live` URL. Stop the cell when you are done.

In [ ]:
%%bash
set -euo pipefail

cd "${EZVC_REPO_DIR}"
export HF_TOKEN="${HF_TOKEN:-}"
export HUGGING_FACE_HUB_TOKEN="${HUGGING_FACE_HUB_TOKEN:-${HF_TOKEN:-}}"
scripts/start_inference_gradio.sh --host 0.0.0.0 --port 7861 --venv .venv --share


## Start training UI

Run this cell instead when you want the fine-tuning UI. Gradio will print a public `gradio.live` URL. Stop the cell when you are done.

In [ ]:
%%bash
set -euo pipefail

cd "${EZVC_REPO_DIR}"
export HF_TOKEN="${HF_TOKEN:-}"
export HUGGING_FACE_HUB_TOKEN="${HUGGING_FACE_HUB_TOKEN:-${HF_TOKEN:-}}"
scripts/start_training_gradio.sh --host 0.0.0.0 --port 7862 --venv .venv --share
